# Build and Evaluate Custom Metrics with OpenAI LLMaaJ Provider for DPTA using One step configuration

This notebook should be run using Runtime 24.1 & Python 3.11 or greater runtime environment. if you are viewing this in Watson Studio and do not see Python 3.11.x in the upper right corner of your screen, please update the runtime now.

This notebook demonstrates how to evaluate custom metrics for detached prompt templates using the LLM as a Judge (LLMaaJ) evaluator, enabling domain-specific evaluation beyond built-in generative AI quality metrics, where a large language model acts as a judge to assess AI-generated responses based on customizable grading logic.

The notebook will create a retrieval augmented generation prompt template asset in a given project, configure OpenScale to monitor that PTA and evaluate custom metrics using OpenAI llmaaj evaluator and model health metrics.

If users wish to execute this notebook for task types other than retrieval_augmented_generation, please consult [this](https://github.com/IBM/watson-openscale-samples/blob/main/IBM%20Cloud/WML/notebooks/watsonx/README.md) document for guidance on evaluating prompt templates for the available task types.

Note : User can search for `EDIT THIS` and fill the inputs needed.

### Contents

- [Package installation and dependencies](#package-installation-and-dependencies)
- [Credentials setup](#credentials-setup)
- [Sample data loading and preview](#sample-data-loading-and-preview)
- [Create facts client](#create-facts-client)
- [Create and configure detached prompt](#create-and-configure-detached-prompt)
    - [Define prompt template](##define-prompt-template)
    - [Create prompt asset](##create-prompt-asset)
- [Create openscale client](#create-openscale-client)
- [Setup LLM as a judge configuration](#setup-llm-as-a-judge-configuration)
    - [Build the OpenAI LLM Provider Configuration](#build-the-openai-llm-provider-configuration)
    - [Define Custom Metrics](#define-custom-metrics)
    - [Set Up the Custom Monitor (One-Step Configuration)](#set-up-the-custom-monitor-one-step-configuration)
- [Risk evaluation](#risk-evaluation)
- [View results](#view-results)
    - [Display metric scores](#display-metric-scores)
    - [Display record-level metrics](#display-record-level-metrics)
    - [View Factsheet information for project subscription](#view-factsheet-information-for-project-subscription)

## Prerequisite

* It requires service credentials for IBM Watson OpenScale:
* Requires a CSV file containing the test data that needs to be evaluated
* Requires the ID of project and space in which you want to create the prompt template asset.

# Package installation and dependencies

Install all necessary Python packages for this notebook.
- `ibm-aigov-facts-client`: For factsheets
- `ibm-watson-openscale`: IBM OpenScale SDK
  
**Restart the kernel after installation before continuing.**

In [1]:
!pip install --upgrade ibm-aigov-facts-client | tail -n 1
!pip install --upgrade ibm-watson-openscale | tail -n 1


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


# Credentials setup

Configure authentication credentials and environment settings:

- Set `use_cpd = False` when using **IBM Cloud** or `True` when using **Cloud Pak for Data (CPD)**.

**For Cloud:**
- `IAM_URL`: IBM Cloud IAM URL for authentication
- `SERVICE_URL`: Watson OpenScale service URL
- `CLOUD_API_KEY`: IBM Cloud API key for authentication
- `DATAPLATFORM_URL`: IBM Cloud Data Platform URL

**For Cloud Pak for Data (CPD):**
- `CPD_URL`: Cloud Pak for Data cluster URL
- `CPD_USERNAME`, `CPD_PASSWORD`: Authentication credentials
- `CPD_API_KEY`: Optional API key for authentication (alternative to username/password)
- `CPD_VERSION`: Cloud Pak for Data version (e.g., "5.3")
- `SERVICE_INSTANCE_ID`: Watson OpenScale datamart identifier

**Project and Judge model provider credentials:**
- `PROJECT_ID`: Watson Studio project ID where prompt template will be created
- `PROVIDER_TYPE`: Judge model provider type (e.g., "openai")
- `MODEL_ID`: Specific judge model identifier (e.g., "gpt-4.1-mini")
- `OPENAI_API_KEY`: OpenAI API key for judge model authentication
- `CUSTOM_MONITOR_NAME`: Unique identifier for the custom monitor


In [2]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
use_cpd = False         # set use_cpd = Ture if using CPD cluster

In [ ]:
if use_cpd:
    # ── Cloud Pak for Data credentials ─────────────────────────────────────────────
    CPD_URL      = "<EDIT THIS>"           # e.g. https://cpd-cpd-instance.apps.example.com
    CPD_USERNAME = "cpadmin"               # e.g. "cpadmin"
    CPD_PASSWORD = "<EDIT THIS>"
    # CPD_API_KEY  = "<EDIT THIS>"         # Optional: If using API key instead of username/password
    CPD_VERSION  = "5.3"
    SERVICE_INSTANCE_ID = "00000000-0000-0000-0000-000000000000"    # e.g. "00000000-0000-0000-0000-000000000000"
else:
    # ── WatsonX cloud credentials ─────────────────────────────────────────────
    IAM_URL = "https://iam.cloud.ibm.com"
    SERVICE_URL = "https://api.aiopenscale.cloud.ibm.com"
    CLOUD_API_KEY = "<EDIT THIS>"                   # YOUR_CLOUD_API_KEY
    DATAPLATFORM_URL = "https://dataplatform.cloud.ibm.com"

In [ ]:
# ── deployment space and judge model settings ─────────────────────────────────────────
PROJECT_ID = "<EDIT THIS>"            # Watson Studio deployment space ID
PROVIDER_TYPE = "openai"              # Type of judge model provider
MODEL_ID = "gpt-4.1-mini"             # Judge model used for LLM-based evaluation, such as "gpt-5.4", "gpt-4.1", "gpt-4.1-mini", etc.

In [6]:
CUSTOM_MONITOR_NAME = "rag_quality"  # Set custom monitor definition name

In [ ]:
# Enter OPENAI API KEY
OPENAI_API_KEY  = "<EDIT THIS>"

### Create the access token

Generate a temporary access token from CPD credentials for API authentication. The token expires after a certain period; re-run this cell if you encounter authentication errors.

In [8]:
import requests, json

def generate_access_token():
    headers={}
    if not use_cpd: 
        headers["Content-Type"] = "application/x-www-form-urlencoded"
        headers["Accept"] = "application/json"
        data = {
            "grant_type": "urn:ibm:params:oauth:grant-type:apikey",
            "apikey": CLOUD_API_KEY,
            "response_type": "cloud_iam"
        }
        response = requests.post(IAM_URL + "/identity/token", data=data, headers=headers)
        json_data = response.json()
        iam_access_token = json_data["access_token"]
    else:
        headers["Content-Type"] = "application/json"
        headers["Accept"] = "application/json"
        data = {
            "username": CPD_USERNAME,
            "password": CPD_PASSWORD
        }
        data = json.dumps(data).encode("utf-8")
        url = CPD_URL + "/icp4d-api/v1/authorize"
        response = requests.post(url=url, data=data, headers=headers, verify=False)
        json_data = response.json()
        iam_access_token = json_data["token"]
        
    return iam_access_token

iam_access_token = generate_access_token()

print("✅ Access token generated.")

✅ Access token generated.


# Sample data loading and preview

Load the RAG evaluation dataset from CSV. The dataset must include:
- `question`: User queries
- `answer`: Reference/ground truth answers
- `generated_text`: Model-generated responses (pre-computed)
- `context1`, `context2`, `context3`: Retrieved context documents for RAG


In [ ]:
# Download sample RAG test data (insurance FAQ dataset).
# Replace this URL with your own data source if needed.
!rm -fr rag_state_union_gt.csv
!wget "https://raw.githubusercontent.com/IBM/watson-openscale-samples/main/IBM%20Cloud/WML/assets/data/watsonx/rag_state_union_gt.csv"

In [10]:
import pandas as pd

TEST_DATA_PATH = "rag_state_union_gt.csv"

# Load the data
llm_data = pd.read_csv(TEST_DATA_PATH)

# Optional: limit to first N rows for a quick test run
llm_data = llm_data.head(5)
llm_data.to_csv(TEST_DATA_PATH, index=False)

print(f"✅ Loaded {len(llm_data)} rows. Columns: {list(llm_data.columns)}")
llm_data

✅ Loaded 5 rows. Columns: ['context1', 'context2', 'context3', 'context4', 'question', 'answer', 'generated_text']


,context1,context2,context3,context4,question,answer,generated_text
0,"Last month, I announced our plan to supercharg...",For that purpose we’ve mobilized American grou...,"If you travel 20 miles east of Columbus, Ohio,...",But cancer from prolonged exposure to burn pit...,What is ARPA-H?,ARPA-H is the Advanced Research Projects Agenc...,ARPA-H stands for Advanced Research Projects A...
1,So let’s not wait any longer. Send it to my de...,"If you travel 20 miles east of Columbus, Ohio,...",When we use taxpayer dollars to rebuild Americ...,It is going to transform America and put us on...,What is the investment of Ford and GM to build...,Ford is investing $11 billion to build electri...,Ford is investing $11 billion to build electri...
2,My plan will cut the cost in half for most fam...,We got more than 130 countries to agree on a g...,And unlike the $2 Trillion tax cut passed in t...,We’re going after the criminals who stole bill...,What is the proposed tax rate for corporations?,The proposed tax rate for corporations is a 15...,15% minimum tax rate for corporations. The gl...
3,"If you travel 20 miles east of Columbus, Ohio,...",So let’s not wait any longer. Send it to my de...,When we use taxpayer dollars to rebuild Americ...,It is going to transform America and put us on...,What is Intel going to build?,Intel is going to build a $20 billion semicond...,semiconductor “mega site” with up to eight sta...
4,So let’s not wait any longer. Send it to my de...,"If you travel 20 miles east of Columbus, Ohio,...",It is going to transform America and put us on...,Vice President Harris and I ran for office wit...,How many electric vehicle charging stations ar...,The document does not provide information on t...,"500,000 Answer: 500,000 electric vehicle charg..."


# Create facts client

Initialize the AI Governance Facts Client for managing prompt template assets and factsheets. Configuration:
- `cloud_pak_for_data_configs`: CPD authentication credentials
- `container_id`: PROJECT_ID where assets will be created
- `container_type`: "project" for project-based assets
- `disable_tracing`: True to disable telemetry collection

In [11]:
from ibm_aigov_facts_client import AIGovFactsClient, CloudPakforDataConfig

if not use_cpd:
    facts_client = AIGovFactsClient(
        api_key=CLOUD_API_KEY,
        container_id=PROJECT_ID,
        container_type="project",
        disable_tracing=True
    )
else:
    from ibm_aigov_facts_client import CloudPakforDataConfig
    creds=CloudPakforDataConfig(
        service_url=CPD_URL,
        username=CPD_USERNAME,
        password=CPD_PASSWORD
    )
    facts_client = AIGovFactsClient(
        cloud_pak_for_data_configs=creds,
        container_id=PROJECT_ID,
        container_type="project",
        disable_tracing=True
    )

print("✅ AIGovFactsClient initialized.")

✅ AIGovFactsClient initialized.


# Create and configure detached prompt

This section demonstrates how to create and monitor a prompt template asset in a Watson Studio project. You'll:
1. Define the prompt template structure
2. Create the detached prompt asset in the project
3. Configure OpenScale monitoring

## Define prompt template

Define the prompt structure for RAG (Retrieval Augmented Generation) tasks. The template includes:
- System instructions defining the assistant's role and behavior
- Context placeholders (`{context1}`, `{context2}`, `{context3}`) for retrieved documents
- Question placeholder (`{question}`) for user queries
- Guidelines for generating accurate, context-based responses

In [12]:
# The prompt template used by the external RAG model.
# Variables like {context1}, {question} map to columns in your test data.
prompt_input = """[INST] <>You are a knowledgeable assistant that answers questions about U.S. policy, government initiatives, 
and State of the Union addresses. You must provide accurate, factual answers based solely on the provided context documents.
If the answer cannot be found in the context, clearly state that you don't have that information.<>

Context Documents:
{context1}
{context2}
{context3}

Instructions:
- Answer questions using ONLY information from the provided context documents
- Provide specific details like numbers, names, and facts when available
- If the answer is not in the context, respond: "I don't have enough information in the provided context to answer that question."
- Keep answers concise and factual
- Do not make assumptions or add information not present in the context
- When multiple contexts contain relevant information, synthesize them coherently

Question: {question} [/INST]

Step 1: Identify relevant information in the provided contexts
Step 2: Extract specific facts, figures, and details that answer the question
Step 3: Formulate a clear, accurate answer based only on the context
"""

## Create prompt asset

Create a detached prompt template asset in the project. Configuration includes:
- `detached_information`: External model details
  - `prompt_id`: Unique identifier for the detached prompt
  - `model_id`: External LLM model identifier (e.g., "meta-llama/llama-3-70b-instruct")
  - `model_provider`: Model provider name (e.g., "Facebook")
  - `model_url`: External model endpoint URL
- `prompt_template`: Template configuration
  - `input`: The prompt text with variable placeholders
  - `prompt_variables`: Dictionary mapping variable names to empty strings (values filled at runtime)
- `task_id`: "retrieval_augmented_generation" defines the RAG task type
- Returns `project_pta_id`: Unique identifier for the created prompt template asset

**Note:** "Detached" indicates the LLM is hosted externally, not in Watson ML

In [13]:
from ibm_aigov_facts_client import DetachedPromptTemplate, PromptTemplate

# ── Describe the external model ───────────────────────────────────────────────
detached_information = DetachedPromptTemplate(
    prompt_id="detached_prompt_rag",
    model_id="meta-llama/llama-3-70b-instruct",     # The external model's ID
    model_provider="Facebook",                      # The model provider
    model_name="llama-3-70b-instruct",
    model_url="https://huggingface.co/google/flan-t5-base",     # The external model's endpoint URL
    prompt_url="prompt_url",                         # The prompt URL (if applicable)
    prompt_additional_info={"model_owner": "huggingface"}
)

# ── Define the prompt template variables ─────────────────────────────────────
# These must match the column names in your test data
prompt_template = PromptTemplate(
    input=prompt_input,
    prompt_variables={
        "context1": "",
        "context2": "",
        "context3": "",
        "question": ""
    },
    input_prefix="",
    output_prefix=""
)

# ── Create the Detached PTA ───────────────────────────────────────────────────
pta_details = facts_client.assets.create_detached_prompt(
    model_id="meta-llama/llama-3-70b-instruct",
    task_id="retrieval_augmented_generation",
    name="Detached PTA for RAG with LLMaaJ Custom Metrics",
    description="Detached prompt template for RAG evaluation with LLM as a Judge custom metrics on CPD",
    prompt_details=prompt_template,
    detached_information=detached_information
)

project_pta_id = pta_details.to_dict()["asset_id"]
print(f"✅ Detached PTA created. ID: {project_pta_id}")

2026/04/20 09:52:51 INFO : ------------------------------ Detached Prompt Creation Started ------------------------------
2026/04/20 09:52:52 INFO : The detached prompt with ID a60754d5-db54-4433-98c5-af1c1d819dc2 was created successfully in container_id 16b10567-e64c-4238-ae9c-3967f582ed62.
✅ Detached PTA created. ID: a60754d5-db54-4433-98c5-af1c1d819dc2


# Create openscale client

Initialize the Watson OpenScale client for model monitoring and governance.

In [14]:
from ibm_cloud_sdk_core.authenticators import IAMAuthenticator, CloudPakForDataAuthenticator
from ibm_watson_openscale import APIClient


if use_cpd:
    authenticator = CloudPakForDataAuthenticator(
        url=CPD_URL,
        username=CPD_USERNAME,
        password=CPD_PASSWORD,
        disable_ssl_verification=True
    )
    wos_client = APIClient(
        service_url=CPD_URL,
        authenticator=authenticator,
        service_instance_id=SERVICE_INSTANCE_ID
    )

else:
    service_instance_id = None  # Update this to refer to a particular service instance
    authenticator = IAMAuthenticator(
        apikey=CLOUD_API_KEY,
        url=IAM_URL
    )
    wos_client = APIClient(
        authenticator=authenticator,
        service_url=SERVICE_URL,
        service_instance_id=service_instance_id
    )
    data_mart_id = wos_client.service_instance_id

print(f"✅ Watson OpenScale client initialized. SDK version: {wos_client.version}")

[Warning] No region provided : Using default region as us-south
✅ Watson OpenScale client initialized. SDK version: 3.1.5


## Map openscale instance to project

When the authentication is on CPD then we need to add additional step of mapping the `project_id` to an OpenScale instance.

In [15]:
from ibm_watson_openscale.base_classes import ApiRequestFailure

if use_cpd:
    try:
        wos_client.wos.add_instance_mapping(
            service_instance_id=SERVICE_INSTANCE_ID, project_id=PROJECT_ID
        )
    except ApiRequestFailure as arf:
        if arf.response.status_code == 409:
            # Instance mapping already exists
            pass
        else:
            raise arf

# Setup LLM as a judge configuration

### Build the OpenAI LLM Provider Configuration

Configure the LLM that will act as the judge to evaluate summarization outputs. The configuration includes:
- `API_KEY`: OpenAI API key for authentication
- `type`: Provider type (e.g., "openai")
- `MODEL_NAME`: Specific model identifier (e.g., "gpt-4", "gpt-3.5-turbo")

Note: For supported evaluator types, refer to: https://github.com/IBM/watson-openscale-samples/wiki/Generative-AI-Evaluator-templates#supported-evaluator-types

In [16]:
# Configuration for the judge model (the LLM that evaluates your Summarization outputs)
LLM_PROVIDER_CONFIG = {
    "API_KEY":      OPENAI_API_KEY,
    "type":         PROVIDER_TYPE,
    "MODEL_NAME":   MODEL_ID
}

print("✅ LLM provider config ready.")

✅ LLM provider config ready.


### Define Custom Metrics
**prompt:**
- Defines the instruction provided to the LLM judge describing how a response should be evaluated.  
- The prompt contains placeholders that will later be replaced with values from the evaluation dataset during metric computation.

**grading_options:**
- Defines the scoring scale used by the LLM judge. 
- Each option represents a possible evaluation outcome and is associated with a numeric score that will be recorded as the metric result.

**grader_prompt_variables_mapping:**
- Maps placeholders used inside the evaluation prompt to the corresponding column names in the dataset.  
- This mapping ensures that the correct dataset values are substituted into the prompt before it is sent to the LLM judge for evaluation.

Each metric configuration includes:
- `name`: Metric identifier
- `description`: Explanation of what the metric evaluates
- `thresholds`: Acceptable score ranges (`upper_limit`, `lower_limit`)
- `computation`: Evaluation logic
  - `prompt`: Grading instructions for the judge LLM with variable placeholders
  - `grading_options`: Possible evaluation outcomes with assigned scores
    - `name`: Option identifier
    - `description`: What this option represents
    - `value`: Numeric score (0-1 scale)
- `grader_prompt_variables_mapping`: Maps prompt variables to CSV columns
- `thresholds`: Acceptable value ranges
  - `upper_limit`: Maximum acceptable value
  - `lower_limit`: Minimum acceptable value
- `dataset_type`: Data source for evaluation, supported data types are `payload_logging` and `feedback`
  - `feedback`: Uses reference `answer` column (ground truth)


In [17]:
# ── Custom Metrics ────────────────────────────────────────────────────
# Each metric uses an LLM grader prompt to score a quality dimension.

MONITOR_METRICS = [

    # ── Metric 1: Answer Completeness ────────────────────────────────
    # Checks whether the generated answer fully addresses the question.
    # Uses the 'feedback' dataset (compares question vs. reference answer).
    {
        "name": "answer_completeness",
        "description": "Evaluates whether the AI-generated response fully addresses the question.",
        "computation": {
            "prompt": (
                "You are an expert grader. Your job is to evaluate the completeness of an "
                "AI-generated response based on the user question.\n\n"
                "**Question:**\n{input}\n\n"
                "**AI-Generated Response:**\n{output}\n\n"
                "Determine whether the response is complete using the grading scale below.\n\n"
                "## Grading Scale:\n"
                "complete   - The response thoroughly addresses all parts of the question.\n"
                "partial    - The response addresses some parts but is missing key information.\n"
                "incomplete - The response fails to address the question."
            ),
            "grading_options": [
                {"name": "complete",   "value": 1},
                {"name": "partial",    "value": 0.5},
                {"name": "incomplete", "value": 0}
            ]
        },
        # Maps prompt variables to dataset columns
        "grader_prompt_variables_mapping": {
            "input": "question",   # {input} → "question" column
            "output": "answer"     # {output} → "answer" column
        },
        "dataset_type": "feedback",  # Uses the reference answer column
        "thresholds": {
            "lower_limit": 0.4
        }
    },

    # ── Metric 2: Conciseness ────────────────────────────────────────
    # Checks whether the generated answer is concise and to the point.
    # Uses the 'payload_logging' dataset (evaluates model-generated output).
    {
        "name": "conciseness",
        "description": "Evaluates whether the AI-generated response is concise and to the point.",
        "computation": {
            "prompt": (
                "You are evaluating the conciseness of an AI-generated answer.\n\n"
                "Question:\n{question}\n\n"
                "Answer:\n{answer}\n\n"
                "A concise answer should:\n"
                "- Directly address the question\n"
                "- Contain only essential information\n"
                "- Avoid repetition or unnecessary context\n"
                "- Be clear and brief\n\n"
                "Based on these criteria, classify the answer into one of the following categories:\n"
                "- highly_concise\n"
                "- moderately_concise\n"
                "- not_concise\n\n"
                "Return only the category name."
            ),
            "grading_options": [
                {
                    "name": "highly_concise",
                    "description": "The answer directly addresses the question and contains only essential information.",
                    "value": 1
                },
                {   
                    "name": "moderately_concise",
                    "description": "The answer addresses the question but includes some extra details.",
                    "value": 0.5
                },
                {   
                    "name": "not_concise",
                    "description": "The answer contains excessive or irrelevant information.",
                    "value": 0
                }
            ]
        },
        "grader_prompt_variables_mapping": {    
            "question": "question",
            "answer": "answer"
        },
        "dataset_type": "feedback",
        "thresholds": { 
            "lower_limit": 0.4
        }
    }
]

#### LLM Judge Configuration

Complete configuration for LLM-as-a-Judge custom monitoring:
- **DATAMART_ID**: Watson OpenScale service instance ID
- **LLM_PROVIDER_CONFIG**: Judge model configuration (defined above)
- **PROMPT_TEMPLATE_ASSET_ID**: Created prompt template asset ID
- **PROJECT_ID**: Watson Studio project ID
- **operational_space_id:** Set to `"development"`
  - Use `"development"` for staging or testing environments.
- **PROBLEM_TYPE**: Specifies the problem type being monitored. e.g: "retrieval_augmented_generation"
- **MIN_RECORDS**, **MAX_RECORDS**: Evaluation record limits
- **CUSTOM_MONITOR_CONFIG**: Custom monitor settings
  - `CUSTOM_MONITOR_NAME`: Monitor definition identifier
  - `MONITOR_METRICS`: List of custom metrics (defined above)
- **DELETE_LLM_PROVIDER**: Set to `False` if don't want to delete llm provider, default value is `True`
- **DELETE_CUSTOM_MONITOR**: Set to `False` if don't want to delete custom monitor, default value is `True`

In [18]:
llmaj_config = {

    # ── Watson OpenScale data mart ────────────────────────────────────────────
    "DATAMART_ID": data_mart_id,

    # ── Judge model ───────────────────────────────────────────────────────────
    "LLM_PROVIDER_CONFIG": LLM_PROVIDER_CONFIG,

    # ── Prompt template asset ─────────────────────────────────────────────────
    "PROMPT_TEMPLATE_ASSET_ID": project_pta_id,
    "PROJECT_ID":               PROJECT_ID,

    "LABEL_COLUMN":         "answer",                       # Reference answer column
    "OPERATIONAL_SPACE_ID": "development",                  # Use "development" for project-based PTAs
    "PROBLEM_TYPE":         "retrieval_augmented_generation",

    # ── RAG-specific field mappings ───────────────────────────────────────────
    "CONTEXT_FIELDS":  ["context1", "context2", "context3"],
    "QUESTION_FIELD":  "question",

    # ── Evaluation record limits ──────────────────────────────────────────────
    "MIN_RECORDS": 5,

    # ── Custom monitor definition ─────────────────────────────────────────────
    "CUSTOM_MONITOR_CONFIG": {
        "CUSTOM_MONITOR_NAME": CUSTOM_MONITOR_NAME,
        "MONITOR_METRICS":     MONITOR_METRICS
    },

    # ── Cleanup options ───────────────────────────────────────────────────────
    # Set to True to delete and recreate the monitor if it already exists
    "DELETE_LLM_PROVIDER":  True,
    "DELETE_CUSTOM_MONITOR": True
}

## Set Up the Custom Monitor (One-Step Configuration)

This single SDK call automates the entire setup process:

1. Creates/Reuses LLM Provider
2. Creates Custom Monitor Definition
3. Creates Subscription
4. Creates Monitor Instance
5. Creates Custom Dataset

**Note:** If you run this cell multiple times, existing resources will be deleted and recreated (controlled by `DELETE_CUSTOM_MONITOR` and `DELETE_LLM_PROVIDER` flags in the config).

In [19]:
# Set up the LLM Judge configuration — this creates all required resources
result = wos_client.custom_monitor.setup_llm_judge_configuration(config=llmaj_config)
result

Deleting custom dataset: 019da8d6-4677-7495-b5a3-bd420039f4b9 (table: YPPROD_c2e6a56f_ea61_4348_95e3_c5c8923afc59_PL_PROD.rag_quality_019da8d6-2d57-730f-83ea-c30474d40740)



 Waiting for end of deleting data set 019da8d6-4677-7495-b5a3-bd420039f4b9 




finished

-----------------------------------------
 Successfully finished deleting data set 
-----------------------------------------


Custom dataset 019da8d6-4677-7495-b5a3-bd420039f4b9 deleted successfully



 Waiting for end of deleting monitor instance 019da8d6-4364-7538-8905-c44256800329 




finished

-------------------------------------------------
 Successfully finished deleting monitor instance 
-------------------------------------------------





 Waiting for end of deleting monitor definition rag_quality 




finished

---------------------------------------------------
 Successfully finished deleting monitor definition 
---------------------------------------------------





 Waiting for end of adding monitor definit

{'custom_monitor_id': 'rag_quality',
 'llm_provider_id': '019da921-12b3-7991-a80c-28d9fe27d468',
 'llm_provider_name': 'LLM_Evaluator_watsonx_c2e6a56f-ea61-4348-95e3-c5c8923afc59',
 'llm_provider': 'openai',
 'subscription_id': '019da921-2483-7559-adda-f172fd4b8c1a',
 'service_provider_id': '019da8d6-2a6e-7f36-a600-cd684a3b285c',
 'monitor_instance_id': '019da921-3271-7abd-8d79-ec31c7fec5a7',
 'custom_dataset_id': '019da921-356e-7f4b-a9d1-d75a285c84a7',
 'custom_dataset_table_name': 'YPPROD_c2e6a56f_ea61_4348_95e3_c5c8923afc59_PL_PROD.rag_quality_019da921-2483-7559-adda-f172fd4b8c1a',
 'prompt_setup_response': {'subscription_id': '019da921-2483-7559-adda-f172fd4b8c1a',
  'service_provider_id': '019da8d6-2a6e-7f36-a600-cd684a3b285c',
  'status': {'state': 'FINISHED'},
  'full_response': {'prompt_template_asset_id': 'a60754d5-db54-4433-98c5-af1c1d819dc2',
   'project_id': '16b10567-e64c-4238-ae9c-3967f582ed62',
   'service_provider_id': '019da8d6-2a6e-7f36-a600-cd684a3b285c',
   'subscri

In [20]:
# Extract the key IDs from the result
monitor_instance_id     = result["monitor_instance_id"]
subscription_id         = result["prompt_setup_response"]["subscription_id"]
mrm_monitor_instance_id = result["prompt_setup_response"]["full_response"]["mrm_monitor_instance_id"]
custom_dataset_id       = result["custom_dataset_id"]  # For record-level metrics

print(f"   Custom Monitor ID:    {result['custom_monitor_id']}")
print(f"   Monitor Instance ID:  {monitor_instance_id}")
print(f"   LLM Provider ID:      {result['llm_provider_id']}")
print(f"   Subscription ID:      {subscription_id}")
print(f"   Custom Dataset ID:    {custom_dataset_id}")

   Custom Monitor ID:    rag_quality
   Monitor Instance ID:  019da921-3271-7abd-8d79-ec31c7fec5a7
   LLM Provider ID:      019da921-12b3-7991-a80c-28d9fe27d468
   Subscription ID:      019da921-2483-7559-adda-f172fd4b8c1a
   Custom Dataset ID:    019da921-356e-7f4b-a9d1-d75a285c84a7


# Risk evaluation

Execute risk evaluation on the RAG model using the test dataset. This process:
- Uploads the CSV test data to OpenScale
- Triggers all configured monitors (including custom LLM-as-a-Judge metrics)
- Computes metric scores for each record and aggregated results
- Returns evaluation status and detailed results

In [21]:
#####################################################################################
######### For pre_production flow
######################################################################################
response = wos_client.monitor_instances.mrm.evaluate_risk(
    monitor_instance_id=mrm_monitor_instance_id,
    test_data_set_name="rag_state_union_data",
    test_data_path=TEST_DATA_PATH,
    content_type="multipart/form-data",
    body={},
    project_id=PROJECT_ID,
    includes_model_output=True,   # True because generated_text is already in the CSV
    background_mode=False         # Wait for completion before returning
)
eval_result = response.result.to_dict()
print(f"\n✅ Evaluation status: {eval_result['entity']['status']['state']}")
eval_result




 Waiting for risk evaluation of MRM monitor 019da921-4b35-7793-be9e-6b1ec2f5173c 




upload_in_progress.
running.
finished

---------------------------------------
 Successfully finished evaluating risk 
---------------------------------------



✅ Evaluation status: finished


{'metadata': {'id': 'dfd00a2a-831f-421c-b99b-fafbd547aa67',
  'created_at': '2026-04-20T04:24:08.348Z',
  'created_by': 'iam-ServiceId-b317a8da-d926-496e-b0ca-6bcc57f556ae'},
 'entity': {'triggered_by': 'user',
  'parameters': {'evaluation_start_time': '2026-04-20T04:23:55.268503Z',
   'evaluator_user_key': '4341b7e5-7c0a-44ff-9026-38fd4fcce12f',
   'facts': {'state': 'finished'},
   'is_auto_evaluated': False,
   'measurement_id': '019da921-a803-71dd-9548-590c670081fa',
   'monitors_run_status': [{'monitor_id': 'model_health',
     'status': {'state': 'finished'}},
    {'monitor_id': 'rag_quality', 'status': {'state': 'finished'}}],
   'project_id': '16b10567-e64c-4238-ae9c-3967f582ed62',
   'prompt_template_asset_id': 'a60754d5-db54-4433-98c5-af1c1d819dc2',
   'user_iam_id': 'IBMid-6930001NUJ',
   'wos_created_deployment_id': '313a8bc9-b0b5-5552-907f-2ccae0f869fe',
   'publish_metrics': 'false',
   'evaluation_tests': ['drift_v2',
    'fairness',
    'generative_ai_quality',
    'gen

# View results

## Display metric scores

The table below shows the **aggregated scores** for each custom metric across all test records.

In [22]:
# Display the computed custom metrics in a table
wos_client.monitor_instances.show_metrics(monitor_instance_id=monitor_instance_id)

2026-04-20 04:24:19.457028+00:00,answer_completeness,019da921-d301-72c7-abea-dedef2dee7bc,0.8,0.4,None,['computed_on:feedback'],rag_quality,019da921-3271-7abd-8d79-ec31c7fec5a7,33bf84de-4d92-44a2-ac02-d4faea1bdbd0,subscription,019da921-2483-7559-adda-f172fd4b8c1a
2026-04-20 04:24:19.457028+00:00,conciseness,019da921-d301-72c7-abea-dedef2dee7bc,0.7,0.4,None,['computed_on:feedback'],rag_quality,019da921-3271-7abd-8d79-ec31c7fec5a7,33bf84de-4d92-44a2-ac02-d4faea1bdbd0,subscription,019da921-2483-7559-adda-f172fd4b8c1a


## Display record-level metrics

View the individual metric scores for each record in your test data.

In [23]:
# Display record-level metrics for each row in the test data
if custom_dataset_id:
    wos_client.data_sets.show_records(data_set_id=custom_dataset_id)
else:
    print("⚠️  Custom dataset ID not available. Record-level metrics cannot be displayed.")

MRM_4c7bf608-e117-49ce-a7f1-871f2afb4349-0,2026-04-20T04:23:57.351Z,feedback,2026-04-20T04:24:23.621359Z,33bf84de-4d92-44a2-ac02-d4faea1bdbd0,0.5,0.5,019da921-302e-7d60-af2c-f618d3305987
MRM_4c7bf608-e117-49ce-a7f1-871f2afb4349-1,2026-04-20T04:23:57.351Z,feedback,2026-04-20T04:24:23.621359Z,33bf84de-4d92-44a2-ac02-d4faea1bdbd0,1.0,0.5,019da921-302e-7d60-af2c-f618d3305987
MRM_4c7bf608-e117-49ce-a7f1-871f2afb4349-2,2026-04-20T04:23:57.351Z,feedback,2026-04-20T04:24:23.621359Z,33bf84de-4d92-44a2-ac02-d4faea1bdbd0,0.5,1.0,019da921-302e-7d60-af2c-f618d3305987
MRM_4c7bf608-e117-49ce-a7f1-871f2afb4349-3,2026-04-20T04:23:57.351Z,feedback,2026-04-20T04:24:23.621359Z,33bf84de-4d92-44a2-ac02-d4faea1bdbd0,1.0,1.0,019da921-302e-7d60-af2c-f618d3305987
MRM_4c7bf608-e117-49ce-a7f1-871f2afb4349-4,2026-04-20T04:23:57.351Z,feedback,2026-04-20T04:24:23.621359Z,33bf84de-4d92-44a2-ac02-d4faea1bdbd0,1.0,0.5,019da921-302e-7d60-af2c-f618d3305987


## View Factsheet information for project subscription

Users can navigate to the project to view the facts published in the factsheet.

In [24]:
if use_cpd:
    factsheets_url = f"{CPD_URL}/wx/prompt-details/{project_pta_id}/factsheet?context=wx&project_id={PROJECT_ID}"
else:
    factsheets_url = factsheets_url = ("{}/wx/prompt-details/{}/factsheet?context=wx&project_id={}".format(DATAPLATFORM_URL, project_pta_id, PROJECT_ID))

print(f"User can navigate to the published facts in project {factsheets_url}")

User can navigate to the published facts in project https://dataplatform.cloud.ibm.com/wx/prompt-details/a60754d5-db54-4433-98c5-af1c1d819dc2/factsheet?context=wx&project_id=16b10567-e64c-4238-ae9c-3967f582ed62


## Congratulations!

You have finished the hands-on lab for IBM Watson OpenScale. You can now navigate to the prompt template asset in your project and click on the Evaluate tab to visualise the results on the UI.